# Full pipeline on Matterport3D Dataset

## Data Loading

In [ ]:
import numpy as np
import open3d as o3d
import trimesh
import os
import sys
sys.path.insert(0, '../')
import drm
import drm.detect
import drm.align
import drm.generate
import drm.pipeline
from PIL import Image
import torch
import numpy as np

%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
plyPath = Path("/home/jvermandere/datasets/MatterPort3D/selection/5LpN3gDmAk7__1f9c76967f3b46ec87c1c58a86094a58/1f9c76967f3b46ec87c1c58a86094a58.ply")
panoPath = plyPath.parent / (plyPath.stem + "_rgb.png")

In [ ]:

pcd = o3d.io.read_point_cloud(plyPath)
pano = Image.open(panoPath)

drm.visualise_open3d([pcd]).show()

## Floor and ceiling Detection

In [ ]:
detected_planes, plane_models, leftovers = drm.detect.detect_planes_iteratively(pcd,min_points=4000, num_iterations=1000, distance_threshold=0.05)

drm.visualise_open3d(detected_planes[2:] + [leftovers], random_color=False).show()

In [ ]:
# Save floor and ceil removed pointcloud for object detection
binPath = drm.open3d_to_bin(detected_planes[2:] + [leftovers], plyPath.parent / (plyPath.stem + "_filtered.bin"))

## Object Detection

### Votenet

In [ ]:
# mmdet3d
import os
demoFile = r"/home/jvermandere/projects/mmdetection3d/demo/pcd_demo.py"
configFile = r"/home/jvermandere/projects/mmdetection3d/configs/votenet/votenet_8xb8_scannet-3d.py"
weightsFile = r"/home/jvermandere/projects/DRM/_weights/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
ScenePath = plyPath.parent / (plyPath.stem + "_filtered.bin")
outDir = plyPath.parent / "votenet_output"
score_thr = 0.3

command = f'python {demoFile} {ScenePath} {configFile} {weightsFile} --pred-score-thr {score_thr} --out-dir {outDir}'
print("running command: " + command)

os.system(command)

In [ ]:
scoreThr = 0.4
bb_detected = (plyPath.parent / "votenet_output" / "preds"/ plyPath.stem).with_stem(plyPath.stem + "_filtered").with_suffix(".json")
detected_meshes, scores, labels = drm.detect.load_detected_boxes_as_mesh(bb_detected, pred_score_thr=scoreThr, labelColors=True, wireframe=False,label_filter=["door", "picture", 'window'], label_filter_inverse=True)
scene = drm.visualise_open3d([pcd])
scene.add_geometry([detected_meshes])
print(scores)
print(labels)
scene.show()

In [ ]:
chosen_boxes,_,_ = drm.detect.load_detected_boxes_as_mesh(bb_detected, pred_score_thr=scoreThr, labelColors=True, wireframe=False,label_filter=["door", "picture", "window"], label_filter_inverse=True)
expanded_boxes = drm.expand_mesh(drm.trimesh_to_open3d(chosen_boxes), offset=0.1)
isolated_pcds, remainder_pcd = drm.detect.split_pointcloud_by_boxes(pcd, expanded_boxes)

scene = drm.visualise_open3d(isolated_pcds + [remainder_pcd], random_color=True)
scene.show()

### Labelcloud

In [ ]:
detected_meshes, labels = drm.detect.load_labelcloud_boxes_as_mesh(plyPath.with_suffix(".json"), labelColors=True)
scene = drm.visualise_open3d([pcd])
scene.add_geometry([detected_meshes])
print(labels)
scene.show()

In [ ]:
expanded_boxes = drm.expand_mesh(drm.trimesh_to_open3d(detected_meshes), offset=0.05)
isolated_pcds, remainder_pcd = drm.detect.split_pointcloud_by_boxes(pcd, expanded_boxes)

scene = drm.visualise_open3d(isolated_pcds + [remainder_pcd], random_color=True)
scene.show()

In [ ]:
o3d.io.write_point_cloud(plyPath.with_name(plyPath.stem + "_empty").with_suffix(".ply"), remainder_pcd)

## Scene Completion

In [ ]:
ramainder_pcd_sub = remainder_pcd.voxel_down_sample(0.05)
scene = drm.visualise_open3d(ramainder_pcd_sub, random_color=False)
scene.show()

In [ ]:
detected_planes, plane_models, leftovers = drm.detect.detect_planes_iteratively(ramainder_pcd_sub,min_points=1000, num_iterations=1000, distance_threshold=0.1)

drm.visualise_open3d(detected_planes, random_color=True).show()

In [ ]:
plane_meshes = drm.generate.create_plane_meshes(plane_models, ramainder_pcd_sub.get_axis_aligned_bounding_box(), max_extend_modifier=1)
print(f"Detected {len(plane_meshes)} planes after clipping.")
filtered_meshes, filtered_pcds = drm.generate.filter_planes_by_points(plane_meshes, detected_planes, distance_threshold=0.1, min_points=1500)

scene = drm.visualise_open3d(filtered_meshes +  filtered_pcds, random_color=True)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

In [ ]:
savelist = [0,1,2,3,4,5]
chosen_filtered_meshes = filtered_meshes # [filtered_meshes[i] for i in savelist]
chosen_filtered_pcds = filtered_pcds # [filtered_pcds[i] for i in savelist]
scene = drm.visualise_open3d(chosen_filtered_meshes + chosen_filtered_pcds, random_color=True)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

In [ ]:
unoccupied = drm.generate.sample_unoccupied_plane_points(chosen_filtered_meshes, chosen_filtered_pcds, voxel_size=0.05)

combined_pcds = [
    pcd + unoccupied_pcd
    for pcd, unoccupied_pcd in zip(chosen_filtered_pcds, unoccupied)
]
cropped_pcds = []
for filtered_pcd in chosen_filtered_pcds:
    cropped_pcd = drm.crop_pointcloud(filtered_pcd, [-20,-20,-20], [20,20,20])
    cropped_pcds.append(cropped_pcd)
# Visualise occupied vs unoccupied side by side
scene = drm.visualise_open3d(cropped_pcds, random_color=False)
for pcd in unoccupied:
    tm = drm.o3d_pointcloud_to_trimesh(pcd)
    if tm is None:
        continue
    tm.colors = np.full((len(tm.vertices), 4), [255, 50, 50, 255], dtype=np.uint8)
    scene.add_geometry(tm)
scene.show()

In [ ]:
projected_meshes = drm.generate.plane_pointclouds_to_meshes(combined_pcds, chosen_filtered_meshes,voxel_size=0.1,scanner_center= [0,0,0])

In [ ]:
scene = drm.visualise_open3d(projected_meshes, random_color=False)
scene.show()

In [ ]:
uv_meshes = drm.generate.assign_plane_uvs(projected_meshes, [0,0,0])
for mesh in uv_meshes:
    mesh.compute_triangle_normals()
textured_meshes = drm.generate.apply_texture_to_planes(uv_meshes, r"/home/jvermandere/projects/DRM/_input/UV_Grid_Sm.jpg")
scene = drm.visualise_open3d(textured_meshes)
scene.lights.append(
    trimesh.scene.lighting.PointLight(
        color=[255, 255, 255, 255],
        intensity=100.0,
        radius=0.0,
    )
)
scene.show()


In [ ]:
def pointcloud_to_pano(
    pcd,
    pano_wh:        tuple[int, int],
    *,
    camera_center:   np.ndarray | None = None,
    camera_rotation: np.ndarray | None = None,
    mode:            str = "color",
    dilation:        int = 0,
) -> np.ndarray | tuple[np.ndarray, np.ndarray]:
    try:
        import open3d as o3d
        if isinstance(pcd, o3d.geometry.PointCloud):
            points = np.asarray(pcd.points, dtype=np.float64)
            colors = (np.asarray(pcd.colors) * 255).astype(np.uint8) if pcd.has_colors() else None
        else:
            pcd = np.asarray(pcd, dtype=np.float64)
            points = pcd[:, :3]
            colors = pcd[:, 3:6].astype(np.uint8) if pcd.shape[1] >= 6 else None
    except ImportError:
        pcd = np.asarray(pcd, dtype=np.float64)
        points = pcd[:, :3]
        colors = pcd[:, 3:6].astype(np.uint8) if pcd.shape[1] >= 6 else None

    W, H = pano_wh

    if camera_center is not None or camera_rotation is not None:
        if camera_center is None or camera_rotation is None:
            raise ValueError("Provide both camera_center and camera_rotation, or neither.")
        C = np.asarray(camera_center,   dtype=np.float64)
        R = np.asarray(camera_rotation, dtype=np.float64)
        points = (points - C) @ R

    X, Y, Z = points[:, 0], points[:, 1], points[:, 2]

    r     = np.sqrt(X**2 + Y**2 + Z**2)
    valid = r > 1e-9

    yaw   = np.where(valid, np.arctan2(Y, X), np.nan)
    pitch = np.where(valid, np.arcsin(np.clip(Z / np.where(valid, r, 1.0), -1.0, 1.0)), np.nan)

    u_norm = (1.0 - yaw   / (2.0 * np.pi)) % 1.0
    v_norm =  0.5 - pitch / np.pi

    px = np.round(u_norm * (W - 1)).astype(np.int32)
    py = np.round(v_norm * (H - 1)).astype(np.int32)

    flat_idx  = py * W + px
    depth_buf = np.full(H * W, np.inf, dtype=np.float32)
    np.minimum.at(depth_buf, flat_idx[valid], r[valid].astype(np.float32))
    depth_buf = depth_buf.reshape(H, W)

    # RGBA: alpha=0 (transparent) for empty pixels, 255 for filled
    color_buf = np.zeros((H, W, 4), dtype=np.uint8)
    if colors is not None:
        winner = valid & (depth_buf.ravel()[flat_idx] == r.astype(np.float32))
        color_buf.reshape(-1, 4)[flat_idx[winner], :3] = colors[winner]
        color_buf.reshape(-1, 4)[flat_idx[winner],  3] = 255

    depth_buf[depth_buf == np.inf] = 0.0

    if dilation > 0:
        from scipy.ndimage import maximum_filter

        kernel_size = 2 * dilation + 1

        if mode in ("depth", "both"):
            depth_buf = maximum_filter(depth_buf, size=kernel_size)

        if colors is not None and mode in ("color", "both"):
            filled_mask  = color_buf[:, :, 3] > 0
            dilated_mask = maximum_filter(filled_mask.astype(np.uint8), size=kernel_size).astype(bool)
            empty_now    = dilated_mask & ~filled_mask

            for c in range(3):
                channel = color_buf[:, :, c].astype(np.float32)
                dilated = maximum_filter(channel, size=kernel_size)
                color_buf[:, :, c] = np.where(empty_now, dilated, color_buf[:, :, c])
            color_buf[:, :, 3] = np.where(dilated_mask, 255, 0)

    if mode == "color":
        return color_buf
    elif mode == "depth":
        return depth_buf
    elif mode == "both":
        return color_buf, depth_buf
    else:
        raise ValueError(f"Unknown mode '{mode}'. Choose 'color', 'depth', or 'both'.")

mask_pcd = o3d.geometry.PointCloud()
for pcd in unoccupied:
    mask_pcd += pcd
mask_pcd.paint_uniform_color([1,0,0])
mask_array = pointcloud_to_pano(remainder_pcd, [2048,1024], dilation=1)
projected_mask_pano = Image.fromarray(mask_array)
projected_mask_pano

In [ ]:
def blend_projection_onto_pano(
    projected: np.ndarray,   # (H, W, 4) RGBA  - output of pointcloud_to_pano
    pano:      np.ndarray,   # (H, W, 3) RGB   - original panorama
) -> np.ndarray:
    """
    Return the pano as RGBA where the transparent regions of projected
    become transparent in the output.
    """
    projected = np.asarray(projected)
    pano      = np.asarray(pano)

    # Add full-opacity alpha to pano
    alpha = np.full((pano.shape[0], pano.shape[1]), 255, dtype=np.uint8)
    result = np.dstack([pano, alpha])   # (H, W, 4)

    # Where projected is transparent, make result transparent too
    transparent = projected[:, :, 3] == 0
    result[transparent, 3] = 0

    return result

masked_pano = Image.fromarray(blend_projection_onto_pano(projected_mask_pano, pano))
masked_pano

In [ ]:
def sample_pano_textures_matterport(
    plane_meshes: list[o3d.geometry.TriangleMesh],
    pano_image:   np.ndarray,
    transform:    np.ndarray,
    base_width:   int = 1024,
) -> list[np.ndarray]:
    """
    Sample a panorama onto each plane mesh using the pano_to_ply.py convention:

        yaw   = (1 - u/W) * 2π   →   u = (1 - yaw/(2π)) % 1
        pitch = (0.5 - v/H) * π  →   v =  0.5 - pitch/π

    Parameters
    ----------
    plane_meshes : list of o3d.geometry.TriangleMesh
    pano_image   : (H, W, 3) or (H, W, 4) uint8 panorama
    transform    : 4×4 camera-to-world matrix built from read_pose():
                       transform[:3, :3] = R   (cam→world rotation)
                       transform[:3,  3] = C   (camera centre in world)
                   Pass np.eye(4) for camera-local point clouds.
    base_width   : texture width in pixels (height scaled by plane aspect ratio)

    Returns
    -------
    list of (H, W, 3) or (H, W, 4) uint8 texture images, one per mesh.
    """
    from scipy.ndimage import map_coordinates

    if not isinstance(pano_image, np.ndarray):
        pano_image = np.array(pano_image)

    pano_h, pano_w, n_channels = pano_image.shape

    R = transform[:3, :3]
    C = transform[:3,  3]

    def world_to_uv(pts_3d: np.ndarray) -> np.ndarray:
        """(N,3) world points → (N,2) normalised UV in [0,1]×[0,1]."""
        pts_cam = (pts_3d - C) @ R          # R^T as right-multiply  (N,3)
        X, Y, Z = pts_cam[:, 0], pts_cam[:, 1], pts_cam[:, 2]
        r       = np.sqrt(X**2 + Y**2 + Z**2)
        valid   = r > 1e-9

        yaw   = np.where(valid, np.arctan2(Y, X), 0.0)
        pitch = np.where(valid, np.arcsin(np.clip(Z / np.where(valid, r, 1.0), -1.0, 1.0)), 0.0)

        u = (1.0 - yaw   / (2.0 * np.pi)) % 1.0
        v =  0.5 - pitch / np.pi

        return np.stack([u, v], axis=-1)    # (N, 2)

    textures = []

    for mesh in plane_meshes:
        verts = np.asarray(mesh.vertices, dtype=np.float64)

        # Build a local 2D coordinate frame on the plane
        centroid = verts.mean(axis=0)
        _, _, vh = np.linalg.svd(verts - centroid)
        normal   = vh[2];  normal /= np.linalg.norm(normal)

        up = np.array([0.0, 0.0, 1.0])
        if abs(np.dot(normal, up)) > 0.9:
            up = np.array([0.0, 1.0, 0.0])
        u_ax = np.cross(up, normal);  u_ax /= np.linalg.norm(u_ax)
        v_ax = np.cross(normal, u_ax); v_ax /= np.linalg.norm(v_ax)
        if np.dot(v_ax, up) < 0:
            v_ax = -v_ax;  u_ax = -u_ax

        proj_u = np.dot(verts - centroid, u_ax)
        proj_v = np.dot(verts - centroid, v_ax)
        u_min, u_max = proj_u.min(), proj_u.max()
        v_min, v_max = proj_v.min(), proj_v.max()
        u_range = u_max - u_min
        v_range = v_max - v_min

        aspect = v_range / u_range if u_range > 0 else 1.0
        tex_w  = base_width
        tex_h  = max(1, int(round(base_width * aspect)))

        # Dense grid of 3D points covering the plane
        px_lin = np.linspace(0.0, 1.0, tex_w)
        py_lin = np.linspace(1.0, 0.0, tex_h)
        uu, vv = np.meshgrid(px_lin, py_lin)

        pts_3d = (
            centroid
            + (uu * u_range + u_min)[..., None] * u_ax
            + (vv * v_range + v_min)[..., None] * v_ax
        ).reshape(-1, 3)

        # Project to panorama UV and sample
        uv     = world_to_uv(pts_3d)
        coords = np.stack([
            uv[:, 1] * (pano_h - 1),
            uv[:, 0] * (pano_w - 1),
        ], axis=0)

        texture = np.zeros((tex_h * tex_w, n_channels), dtype=np.uint8)
        for c in range(n_channels):
            sampled = map_coordinates(
                pano_image[..., c].astype(np.float32),
                coords, order=1, mode="wrap",
            )
            texture[:, c] = sampled.clip(0, 255).astype(np.uint8)

        textures.append(texture.reshape(tex_h, tex_w, n_channels))
        print(f"Plane texture: {tex_w}×{tex_h}px  ({n_channels}ch), "
              f"plane size: {u_range:.2f}×{v_range:.2f}m")

    return textures

masked_plane_textures = sample_pano_textures_matterport(uv_meshes, masked_pano, np.eye(4))
Image.fromarray(masked_plane_textures[0])

In [ ]:
inpainted_textures = drm.generate.inpaint_plane_textures(masked_plane_textures,dilation_px=30)   
Image.fromarray(inpainted_textures[0])

In [ ]:
# Apply to meshes
textured_meshes = drm.generate.apply_texture_to_planes(uv_meshes, inpainted_textures)
scene = drm.visualise_open3d(textured_meshes)
scene.add_geometry(trimesh.creation.axis(0.1))

scene.show()


In [ ]:

scene.export(plyPath.with_name(plyPath.name + "_scene_reconstructed").with_suffix(".glb"))  # open in Blender, three.js, or any glTF viewer